In [ ]:
# ============================================
# 1. Carga PDFs
# ============================================

pdf_paths = [
    "../data/csv/pdf/Anexo_Entrenadores_campeones_de_la_Liga_de_Campeones_de_la_UEFA.pdf",
    "../data/csv/pdf/Anexo_Goleadores_de_la_Liga_de_Campeones_de_la_UEFA.pdf",
    "../data/csv/pdf/Liga_de_Campeones_de_la_UEFA.pdf",
    "../data/csv/pdf/finales_champions_narrativo.pdf",
]

for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs.extend(loader.load())


# ============================================
# 2. Cargar CSVs UEFA (stats)
# Convertir cada fila en texto plano
# ============================================

csv_paths = [
    "../data/csv/csv/ucl_clubs_attacking_stats_1992_2025.csv",
    "../data/csv/csv/ucl_clubs_attempts_stats_1992_2025.csv",
    "../data/csv/csv/ucl_clubs_defending_stats_1992_2025.csv",
    "../data/csv/csv/ucl_clubs_disciplinary_stats_1992_2025.csv",
    "../data/csv/csv/ucl_clubs_distribution_stats_1992_2025.csv",
    "../data/csv/csv/ucl_clubs_goalkeeping_stats_1992_2025.csv",
    "../data/csv/csv/ucl_clubs_goals_stats_1992_2025.csv",
    "../data/csv/csv/ucl_clubs_key_stats_1992_2025.csv",
]

for path in csv_paths:
    df = pd.read_csv(path).astype(str)
    for _, row in df.iterrows():
        texto = "\n".join([f"{col}: {row[col]}" for col in df.columns])
        docs.append(Document(page_content=texto))


# ============================================
# 3. Cargar CSV FINAL DE CHAMPIONS (importante)
# ============================================

df = pd.read_csv("../data/csv/csv/finales_champions.csv").astype(str)

for _, row in df.iterrows():
    texto = f"""
Final de la Champions {row['season_year']}:
Ganador: {row['team_winner']}
Subcampeón: {row['team_runner_up']}
Resultado: {row['score']}

Goleadores del ganador: {row['goleadores_winner']}
Minutos: {row['min_winner']}

Goleadores del subcampeón: {row['goleadores_runner_up']}
Minutos: {row['min_runner_up']}

Estadio: {row['stadium']} ({row['city']}, {row['country']})
"""
    docs.append(Document(page_content=texto))



In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pandas as pd

docs = []

# ======================================================
# 2. Cargar TXT narrativo (una final = un documento)
# ======================================================

txt_path = "../data/csv/txt/finales.txt"

with open(txt_path, "r", encoding="utf-8") as f:
    contenido = f.read()

# Separar por AÑO:
bloques = contenido.split("AÑO: ")

for b in bloques:
    b = b.strip()
    if b == "":
        continue

    # reconstruir el documento entero
    texto = "AÑO: " + b

    docs.append(Document(page_content=texto))




# ============================================
# 4. Split en chunks solo para PDFs y CSVs
# ============================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = []

for d in docs:
    # Si el documento empieza por "AÑO:", es una final → NO chankear
    if d.page_content.startswith("AÑO:"):
        chunks.append(d)
    else:
        # PDFs y CSVs sí se chankean
        partes = splitter.split_documents([d])
        chunks.extend(partes)

print(f"DOCUMENTOS TOTALES: {len(docs)}")
print(f"CHUNKS FINALES: {len(chunks)}")


c:\Users\josit\CUARTO CURSO\APRENDIZAJE AUTOMATICO\Caso-03-IA_GENERATIVA\envIA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DOCUMENTOS TOTALES: 34
CHUNKS FINALES: 34


In [33]:
docs_db = vectordb.get()
print(len(docs_db["documents"]))
print(docs_db["documents"][:3])


34506
['Anexo:Entrenadores campeones de la Liga de\nCampeones de la UEFA\nEsta es la lista de entrenadores campeones de la Copa de Europa y de la Liga de Campeones de la UEFA.\nJosé Villalonga ganó con Real Madrid la primera final de la Copa de Europa en 1956, y repitió la hazaña\nla temporada siguiente.\nLos entrenadores de clubes de Inglaterra dominaron la competición en la década de 1970 y principios de\n1980, ganando seis ediciones consecutivas desde 1977 a 1982. A pesar de esto, los entrenadores de', '1980, ganando seis ediciones consecutivas desde 1977 a 1982. A pesar de esto, los entrenadores de\nnacionalidad italiana, española y alemana han tenido más éxito, ganando doce, once y diez ediciones\ndesde 1956, respectivamente.\nLa competición se convirtió en la Liga de Campeones de la UEFA en 1992,1  con el belga Raymond\nGoethals guiando al Olympique de Marsella al éxito de ese año.\nEl español José Villalonga es el entrenador más joven en ganar una Copa de Europa, con el Real Mad

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)



C:\Users\josit\AppData\Local\Temp\ipykernel_45924\1196791549.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [3]:
from langchain_community.vectorstores import Chroma

vectordb = Chroma.from_documents(
    documents=chunks,             # tus 8584 fragmentos
    embedding=embeddings,         # cómo se convierten a números
    persist_directory="../db_champions"  # carpeta donde se guardará la BD
)

print("Base vectorial creada.")


Base vectorial creada.


In [4]:
retriever = vectordb.as_retriever(search_kwargs={"k": 5})


In [5]:
from langchain_community.llms import Ollama

llm = Ollama(model="llama3")


C:\Users\josit\AppData\Local\Temp\ipykernel_45924\869708315.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3")


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel

prompt = ChatPromptTemplate.from_template("""
Eres un asistente experto en la Champions League.
Usa SOLO el contexto para responder.

Contexto:
{context}

Pregunta:
{question}

Respuesta:
""")

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

resp = rag_chain.invoke("¿Quién marcó en la final de 2006?")
print(resp)


Según el contexto, no se puede determinar quién marcó en la final de 2006 porque ese dato no está presente en las páginas proporcionadas. La información disponible solo hasta 1993 y no hay datos sobre la edición de 2006.


In [54]:
query = "¿Quién marcó en la final de 2005?"

resultados = retriever.invoke({"query": query})

print(f"Se han recuperado {len(resultados)} documentos.\n")

for i, doc in enumerate(resultados):
    print("="*60)
    print(f"📄 Documento {i+1}:")
    print(doc.page_content)
    print("="*60, "\n")


AttributeError: 'dict' object has no attribute 'replace'